In [255]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [256]:
# read raw text file
with open("wonderland.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}\n")

# Look at the first 100 characters
print(text[:100])

Length of dataset in characters: 144009

CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on t


In [257]:
tokens = list(map(int, text.encode("utf-8")))
print(len(tokens))

144009


In [258]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    # in the list of ints (ids) replace all consecutive occurences of pair with new token idx
    newids = []
    i = 0
    while i < len(ids):
        # if we are not at the very last position AND the pair matches then replace it
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [259]:
vocab_size = 512
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (101, 32) into new token 256
merging (32, 116) into new token 257
merging (32, 97) into new token 258
merging (257, 104) into new token 259
merging (105, 110) into new token 260
merging (116, 32) into new token 261
merging (101, 114) into new token 262
merging (100, 32) into new token 263
merging (111, 117) into new token 264
merging (32, 115) into new token 265
merging (259, 256) into new token 266
merging (111, 110) into new token 267
merging (44, 32) into new token 268
merging (260, 103) into new token 269
merging (115, 32) into new token 270
merging (116, 104) into new token 271
merging (101, 110) into new token 272
merging (258, 110) into new token 273
merging (10, 10) into new token 274
merging (111, 32) into new token 275
merging (108, 105) into new token 276
merging (101, 97) into new token 277
merging (104, 97) into new token 278
merging (32, 32) into new token 279
merging (104, 256) into new token 280
merging (111, 114) into new token 281
merging (108, 108) into new t

In [260]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 144009
ids length: 67507
compression ratio: 2.13X


In [261]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

In [262]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:100]) # the first 100 characters

torch.Size([67507]) <built-in method type of Tensor object at 0xffff1013f390>
tensor([ 67,  72,  65,  80,  84,  69,  82,  32,  73, 310,  68, 391, 266,  82,
        406,  98, 287,  45,  72, 111, 387, 274, 336, 357, 428, 260, 110, 269,
        305, 388, 261, 309, 121, 257, 363, 308, 284, 265, 287, 116, 297,  98,
        283, 294, 265, 307, 116, 340, 267, 429,  98, 285, 107, 325, 332, 278,
        118, 297, 333, 271, 269, 305, 412,  58,  32, 267,  99, 256, 281, 257,
        327,  99, 256, 355, 398, 112, 367, 112, 308, 260, 435, 429,  98, 314,
        349, 294, 265, 307, 116, 340, 357, 365, 100, 269, 268, 452, 311, 398,
        110, 275])


In [263]:
# split data into training and validation sets
# use the first 90% for training and the remaining 10% for validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(train_data.shape, data.type)
print(val_data.shape, data.type)

torch.Size([60756]) <built-in method type of Tensor object at 0xffff1013f390>
torch.Size([6751]) <built-in method type of Tensor object at 0xffff1013f390>


In [264]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [265]:
# hyperparameters
batch_size = 1
block_size = 8
max_iters = 50000
eval_interval = 1000
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = vocab_size
# ------------

# torch.manual_seed(1337)

In [266]:
class Model1(nn.Module):

    def __init__(self):
        super().__init__()

        # this model only has a simple embedding table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # size should be (1 x 8)
        B, T = idx.shape 

        # logits should have shape (1 x 8 x vocab_size)
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [267]:
model = Model1()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters()), 'parameters')

262144 parameters


In [268]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [269]:
def generate_output():

    prompt = " "

    context = torch.tensor(
        encode(prompt),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

In [272]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("**** OUTPUT ****")
generate_output()
print("****.       ****")

step 0: train loss 3.7088, val loss 4.0579
step 1000: train loss 3.6695, val loss 4.1596
step 2000: train loss 3.6880, val loss 4.0520
step 3000: train loss 3.6426, val loss 4.1190
step 4000: train loss 3.6475, val loss 4.0410
step 5000: train loss 3.6591, val loss 4.0144
step 6000: train loss 3.6670, val loss 4.0487
step 7000: train loss 3.6203, val loss 4.0624
step 8000: train loss 3.6562, val loss 4.1120
step 9000: train loss 3.6736, val loss 4.0207
step 10000: train loss 3.6211, val loss 4.0360
step 11000: train loss 3.6309, val loss 4.0747
step 12000: train loss 3.6264, val loss 4.1313
step 13000: train loss 3.6023, val loss 4.0634
step 14000: train loss 3.5523, val loss 4.0036
step 15000: train loss 3.6124, val loss 4.0597
step 16000: train loss 3.5767, val loss 3.9813
step 17000: train loss 3.5634, val loss 4.0690
step 18000: train loss 3.5888, val loss 3.9931
step 19000: train loss 3.5793, val loss 4.0173
step 20000: train loss 3.5967, val loss 4.0636
step 21000: train loss 3.5